Fine-tuning PEGASUS for SAMSum dialog dataset

In [ ]:

import torch
from datasets import load_dataset
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForSeq2SeqLM

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load and check dataset
dataset_samsum = load_dataset("knkarthick/samsum")
# training/validation/testing
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")
print(dataset_samsum["test"][0]["dialogue"])
print("\nSummary:")
print(dataset_samsum["test"][0]["summary"])

c:\Users\jia\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jia\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jia\.cache\huggingface\hub\datasets--knkarthick--samsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administra

Split lengths: [14731, 818, 819]
Features: ['id', 'dialogue', 'summary']

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


In [3]:
# Any other chat model will also work - if you're low on memory you can use a smaller one
summarizer = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507")
message_history = [
    {
        "role": "user",
        "content": "Summarize the following text:\n\n[TEXT_TO_SUMMARIZE]"
    }
]
print(summarizer(message_history)[0]["generated_text"][-1]["content"])


c:\Users\jia\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jia\.cache\huggingface\hub\models--Qwen--Qwen3-4B-Instruct-2507. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\jia\AppData\Local\Python\p

I'm ready to help you summarize the text. Please provide the text you'd like me to summarize.


In [ ]:
import torch
from transformers import pipeline

# Any other VLM will also work - if you're low on memory you can use a smaller one
captioner = pipeline("image-text-to-text", model="Qwen/Qwen3-VL-4B-Instruct")
message_history = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS7tYZKl7pEEr2wXPxEdl_CkEdLsFOu4XqMrAdmgpO54kfcAm1guJ7j5L4&s=10",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]
print(captioner(message_history)[0]["generated_text"][-1]["content"])

In [ ]:
from transformers import AutoTokenizer
model_ckpt = "sshleifer/distill-pegasus-cnn-16-4"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

dataset_samsum["train"] = dataset_samsum["train"].select(range(len(dataset_samsum["train"]) // 50))
dataset_samsum["validation"] = dataset_samsum["validation"].select(range(len(dataset_samsum["validation"]) // 50))
dataset_samsum["test"] = dataset_samsum["test"].select(range(len(dataset_samsum["test"]) // 50))

def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch["dialogue"], max_length=1024, truncation=True)
    target_encodings = tokenizer(text_target=example_batch["summary"], max_length=128, truncation=True)
    return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"]
        }

dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched=True)
columns = ["input_ids", "labels", "attention_mask"]
dataset_samsum_pt.set_format(type="torch", columns=columns)

In [ ]:
# Data collator
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
# Trainer
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
output_dir='pegasus-samsum', num_train_epochs=1, warmup_steps=0,
per_device_train_batch_size=1, per_device_eval_batch_size=1,
weight_decay=0.01, logging_steps=10, push_to_hub=False,
eval_strategy='steps', eval_steps=5, save_steps=5e-5,
gradient_accumulation_steps=16)
trainer = Trainer(model=model, args=training_args,
tokenizer=tokenizer, data_collator=seq2seq_data_collator,
train_dataset=dataset_samsum_pt["train"],
eval_dataset=dataset_samsum_pt["validation"])
trainer.train()

In [ ]:
sample_text = dataset_samsum["test"][0]["dialogue"]
reference = dataset_samsum["test"][0]["summary"]
pipe = pipeline("summarization", model=model, tokenizer=tokenizer)
print("Dialogue:")
print(sample_text)
print("\nReference Summary:")
print(reference)
print("\nModel Summary:")
print(pipe(sample_text, length_penalty=0.8, num_beams=8, max_length=64)[0]["summary_text"])